<a href="https://colab.research.google.com/github/EmIbrahimovic/sp26_6_4110_hw_colabs/blob/main/MP02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
### Imports from HW02

import itertools
import numpy as np

class RV:
    """A random variable with a finite domain.

    Example usage:
      A = RV("A", ["x", "y", "z"])
      print(A.domain)
      print(A.dim)
      B = RV("B", [(0, 0), (0, 1), (0, 2)]))
      print(B.domain)
      print(B.dim)
    """

    def __init__(self, name, domain):
        """Initialize a RV.

        Args:
          name: str name for the RV.
          domain: list or tuple of domain values.
        """
        assert isinstance(domain, (list, tuple))
        self.name = name
        self.domain = domain
        self.dim = len(domain)

    def __hash__(self):
        return hash((self.name, tuple(self.domain)))

    def __eq__(self, other):
        return self.name == other.name and self.domain == other.domain

    def __repr__(self):
        return f"RV('{self.name}', {self.domain})"


class Potential:
    """A potential over RVs.

    Example usage:
      A = RV("varA", ["x", "y", "z"])
      B = RV("varB", [0, 1])
      table = np.array([
        [0.1, 0.0],
        [0.4, 0.9],
        [0.5, 0.1]
      ])
      potential = Potential([A, B], table)
      print(potential.rvs)
      print(potential.get(("y", 0)))
      print(potential.get_by_rvs({A: "y", B: 0}))
      print(potential.get_by_names({"varA": "y", "varB": 0}))
    """

    def __init__(self, rvs, table):
        """Create a potential from a list of RVs and a numpy array.

        The order of the random variables corresponds to the axes
        of the numpy array.

        Args:
          rvs: A list or tuple of RVs.
          array: A numpy array of potential values.

        Returns:
          potential: A Potential."""
        assert isinstance(rvs, (tuple, list))
        assert len(rvs) == len(table.shape)
        assert all(rv.dim == dim for (rv, dim) in zip(rvs, table.shape))
        assert isinstance(table, np.ndarray)
        self.rvs = rvs
        self.table = table

    def set(self, assignment, new_value):
        """Given a complete assignment and a value, update table.

        Args:
          assignment: A tuple of values in the order of self.rv.
          new_value: A new value to add to the table.

        Returns:
          value: The value in self.table.
        """
        assert len(assignment) == len(self.rvs)
        indices = [None for _ in self.rvs]
        for index, value in enumerate(assignment):
            rv = self.rvs[index]
            indices[index] = rv.domain.index(value)
        self.table[tuple(indices)] = new_value

    def get(self, assignment):
        """Given a complete assignment of values, lookup table value.

        Args:
          assignment: A tuple of values in the order of self.rv.

        Returns:
          value: The value in self.table.
        """
        assert len(assignment) == len(self.rvs)
        indices = [None for _ in self.rvs]
        for index, value in enumerate(assignment):
            rv = self.rvs[index]
            indices[index] = rv.domain.index(value)
        return self.table[tuple(indices)]

    def get_by_rvs(self, rvs_to_vals):
        """Given a complete assignment of RVs to values, lookup table value.

        Args:
          rvs_to_values: A dict from RVs to values in their domains.

        Returns:
          value: The value in self.table.
        """
        assert set(rvs_to_vals.keys()) == set(self.rvs)
        indices = [None for _ in self.rvs]
        for rv, value in rvs_to_vals.items():
            index = self.rvs.index(rv)
            indices[index] = rv.domain.index(value)
        return self.table[tuple(indices)]

    def get_by_names(self, rv_name_dict):
        """Given a dict from RV names (strs) to assignments,
        return the corresponding value in the potential table.

        Args:
          rv_name_dict: A dict from str names to values.
          potential: A Potential.

        Returns:
          value: The float value from potential.table.
        """
        assert len(rv_name_dict) == len(self.rvs)
        rv_name_to_rv = {rv.name: rv for rv in self.rvs}
        rvs_to_vals = {}
        for rv_name, value in rv_name_dict.items():
            rv = rv_name_to_rv[rv_name]
            rvs_to_vals[rv] = value
        return self.get_by_rvs(rvs_to_vals)

    def __hash__(self):
        return hash(tuple(self.rvs)) ^ hash(self.table.tobytes())

    def __eq__(self, other):
        return hash(self) == hash(other)

    def __neq__(self, other):
        return not (self == other)

    def allclose(self, other, decimals=6):
        """Check whether two potentials are (nearly) equal.
        """
        if set(self.rvs) != set(other.rvs):
            raise ValueError("Can only compare potentials with the same RVs.")
        new_idxs = [other.rvs.index(rv) for rv in self.rvs]
        trans_table2 = np.transpose(other.table, new_idxs)
        assert self.table.shape == trans_table2.shape
        return np.allclose(self.table, trans_table2)


def neighbor_dict(rvs, potentials):
    """This helper function creates a mapping.
      - For each random variable rv, neighbors[rv] is a set of potentials that involve this RV.
      - For each potential pot, neighbors[pot] is a set of random variables involved in this potential.
    """
    neighbors = {v: set() for v in rvs + potentials}
    for p in potentials:
        for v in p.rvs:
            neighbors[p].add(v)
            neighbors[v].add(p)
    return neighbors


def iter_joint_values(rvs):
    """Iterates over joint assignments for a list of RVs.

    Returns an iterator that can be used in a for loop.

    Example usage:
      for assignment in iter_joint_values(rvs):
        print(assignment)  # a tuple
        assert assignment[0] in rvs[0].domain

    Args:
      rvs: A list of RVs.

    Yields:
      assignment: A tuple of ints representing a joint
        assignment of the random variables.
    """
    domains = [rv.domain for rv in rvs]
    return itertools.product(*domains)


def get_sub_assignment(rvs, assignment, sub_rvs):
    """Given an assignment of rvs to values, get a subassignment,
    that is, a sub-tuple of the given assignment involving only
    the given sub_rvs.

    Example usage:
      x = RV("x", [0, 1])
      y = RV("y", ["a", "b"])
      z = RV("z", [3, 5])
      rvs = (x, y, z)
      assignment = (0, "b", 3)
      sub_rvs = (z, x)
      sub_assignment = get_sub_assignment(rvs, assignment, sub_rvs)
      assert sub_assignment == (3, 0)

    Args:
      rvs: A tuple or list of RVs.
      assignment: A tuple or list of values.
      sub_rvs: A tuple or list of RVs, a subset of rvs.

    Returns:
      sub_assignment: A tuple of values.
    """
    assert set(sub_rvs).issubset(set(rvs))
    sub_assignment = []
    for rv in sub_rvs:
        idx = rvs.index(rv)
        val = assignment[idx]
        sub_assignment.append(val)
    return tuple(sub_assignment)


## Part 1.1: Exact Marginalization

Question 1

In [ ]:
# TODO: Implement a function to take in an observation grid, a pairwise potential, and a potential for the
# prior belief of fire at each cell and return the marginal probabilities of fire at each cell

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def make_heatmap(data):
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(data, annot=True, fmt=".3f", cmap="coolwarm",
                vmin=0, vmax=1, xticklabels=False, yticklabels=False, ax=ax)
    ax.set_title("Fire Probability Heat Map")
    return fig

In [ ]:
data = ...
heatmap_fig = make_heatmap(data)
plt.show()

## Part 1.2: Loopy Belief Propagation

Loopy Belief Propagation implementation

In [ ]:
def fire_mrf_lbp_marginals(pairwise_pot, unary_pots, max_iters=100, tol=1e-6):
    """
    Computes approximate marginals using Loopy Belief Propagation given pairwise potential
    and array of unary potentials for each cell.

    Args:
        pairwise_pot: 2x2 array [[phi(0,0), phi(0,1)], [phi(1,0), phi(1,1)]].
        unary_pots: (rows, cols, 2) array of unary potentials for each cell.
        max_iters: int for maximum number of iterations to run LBP for.
        tol: float for how much the marginal probability of fire in any cell can change by between iterations
    """
    rows, cols = unary_pots.shape[0], unary_pots.shape[1]
    # Initialize messages: (Direction, Row, Col, State)
    # 0: Messages from Up, 1: Messages from Down, 2: Messages from Left, 3: Messages from Right
    msgs = np.ones((4, rows, cols, 2))

    for i in range(max_iters):
        old_msgs = msgs.copy()

        # Compute the product of all incoming messages at each node
        # Belief(x) \propto Unary(x) * Message_Up * Message_Down * Message_Left * Message_Right
        combined = unary_pots * msgs[0] * msgs[1] * msgs[2] * msgs[3]

        # Update messages for each direction
        # 1. Update msgs[0] (From Up): Neighbor (r-1) sends to r
        # We need Belief at (r-1) excluding message from r (Down).
        cavity_for_down = combined / msgs[1]
        msgs[0, 1:, :] = np.dot(cavity_for_down[:-1, :], pairwise_pot)

        # 2. Update msgs[1] (From Down): Neighbor (r+1) sends to r
        # We need Belief at (r+1) excluding message from r (Up).
        cavity_for_up = combined / msgs[0]
        msgs[1, :-1, :] = np.dot(cavity_for_up[1:, :], pairwise_pot)

        # 3. Update msgs[2] (From Left): Neighbor (c-1) sends to c
        # We need Belief at (c-1) excluding message from c (Right).
        cavity_for_right = combined / msgs[3]
        msgs[2, :, 1:] = np.dot(cavity_for_right[:, :-1], pairwise_pot)

        # 4. Update msgs[3] (From Right): Neighbor (c+1) sends to c
        # We need Belief at (c+1) excluding message from c (Left).
        cavity_for_left = combined / msgs[2]
        msgs[3, :, :-1] = np.dot(cavity_for_left[:, 1:], pairwise_pot)

        # Normalize messages
        msgs /= np.sum(msgs, axis=-1, keepdims=True)

        # Check for convergence
        if tol and np.allclose(msgs, old_msgs, atol=tol):
            break

    # Final Beliefs
    beliefs = unary_pots * msgs[0] * msgs[1] * msgs[2] * msgs[3]
    marginals = beliefs / np.sum(beliefs, axis=-1, keepdims=True)

    return marginals

Question 5

In [ ]:
def fire_mrf_lbp_conditionals(pairwise_pot, prior, observations, sensor_metrics, max_iters=1000, tol=1e-6):
    """
    Computes approximate marginal probabilities of fire using Loopy Belief Propagation
    given a grid of imperfect observations.

    Args:
        pairwise_pot: 2x2 array [[phi(0,0), phi(0,1)], [phi(1,0), phi(1,1)]].
        prior: Length 2 prior belief potential [prior(clear), prior(fire)]
        observations: 2D rows x cols array with "F", "C", or "U".
        sensor_metrics: [P(obs fire | fire), P(obs fire | clear)].
        max_iters: int for maximum number of iterations to run LBP for.
        tol: float for how much the marginal probability of fire in any cell can change by
             between iterations before it's considered converged
    """
    # TODO: Implement me!
    pass

## Part 1.3: Gibbs Sampling

Question 9

In [ ]:
def fire_mrf_gibbs_conditionals(pairwise_pot, prior, observations, sensor_metrics, num_samples=5000, burn_in=1000):
    """
    Estimates marginal probabilities of fire using Gibbs Sampling.

    Args:
        pairwise_pot: 2x2 array [[phi(0,0), phi(0,1)], [phi(1,0), phi(1,1)]].
        prior: Float prior for fire in a cell.
        observations: 2D rows x cols array with "F", "C", or "U".
        sensor_metrics: [P(obs fire | fire), P(obs fire | clear)]
        num_samples: Total number of iterations to collect.
        burn_in: Number of initial samples to discard
    """
    # TODO: Implement me!
    pass